In [10]:
import random
import statistics

# Eight possible movement directions
DIRS = [
    (-1, 0), (1, 0), (0, -1), (0, 1),
    (-1, -1), (-1, 1), (1, -1), (1, 1)
]


class Courtyard:
    """The environment and its agents: students on a wrapped grid."""

    def __init__(self, n=100, size=30, p=0.3, seed=0):
        self.rng = random.Random(seed)
        self.n, self.size, self.p, self.tick = n, size, p, 0

        # Randomly place students
        self.pos = [
            (self.rng.randrange(size), self.rng.randrange(size))
            for _ in range(n)
        ]

        # Student 0 starts the rumour
        self.knows = [False] * n
        self.knows[0] = True

    def count(self):
        return sum(self.knows)

    def render(self):
        """K = knows, s = unaware, . = empty."""
        rows = [["."] * self.size for _ in range(self.size)]

        for i, (r, c) in enumerate(self.pos):
            if self.knows[i]:
                rows[r][c] = "K"
            elif rows[r][c] == ".":
                rows[r][c] = "s"

        return "\n".join("".join(row) for row in rows)

    def move_all(self):
        """Move each student one cell in a random direction."""
        for i in range(self.n):
            dr, dc = self.rng.choice(DIRS)
            r, c = self.pos[i]

            self.pos[i] = (
                (r + dr) % self.size,
                (c + dc) % self.size
            )

    def next_to_knower(self, cell, know_cells):
        """Check the surrounding cells for a knower."""
        r, c = cell

        return any(
            ((r + a) % self.size, (c + b) % self.size) in know_cells
            for a in (-1, 0, 1)
            for b in (-1, 0, 1)
        )

    def spread(self):
        """Spread the rumour with probability p."""
        know_cells = {
            self.pos[i]
            for i in range(self.n)
            if self.knows[i]
        }

        newly = [
            i for i in range(self.n)
            if not self.knows[i]
            and self.next_to_knower(self.pos[i], know_cells)
            and self.rng.random() < self.p
        ]

        for i in newly:
            self.knows[i] = True

    def step(self):
        """Perform one simulation tick."""
        self.move_all()
        self.spread()
        self.tick += 1

In [11]:
def ticks_until(model, fraction, max_ticks=600):
    target = fraction * model.n

    while model.count() < target and model.tick < max_ticks:
        model.step()

    return model.tick

In [13]:
def average_ticks(p, fraction, seeds=range(30), **kwargs):
    return statistics.mean(
        ticks_until(
            Courtyard(p=p, seed=s, **kwargs),
            fraction
        )
        for s in seeds
    )

In [14]:
print(
    "Half know after",
    round(average_ticks(0.3, 0.5), 1),
    "ticks on average"
)

print(
    "90 percent know after",
    round(average_ticks(0.3, 0.9), 1),
    "ticks on average"
)

runs = [
    ticks_until(Courtyard(p=0.3, seed=s), 0.9)
    for s in range(30)
]

print(
    "Single runs to 90 percent: fastest",
    min(runs),
    ", slowest",
    max(runs)
)

Half know after 43.8 ticks on average
90 percent know after 64.4 ticks on average
Single runs to 90 percent: fastest 50 , slowest 87
